# MURA X-ray Classification: MobileNetV3 vs Vision Transformer

This notebook trains two simple PyTorch models on the MURA X-ray dataset:

- **MobileNetV3**: lightweight CNN baseline
- **Vision Transformer (ViT)**: transformer-based vision model

The task is binary classification:

- `0 = normal`
- `1 = abnormal`

The goal is to keep the code clear and easy to understand before moving to the later medical AI agent phase.


## 1. How to use this notebook

1. Open this notebook in Google Colab.
2. Enable GPU: `Runtime > Change runtime type > T4 GPU`.
3. Create a Kaggle API token from your Kaggle account settings.
4. Run the dataset download cells below and upload `kaggle.json` when Colab asks for it.
5. Run the notebook from top to bottom.
6. Start with small settings. Later, increase epochs and remove sample limits for the final experiment.

By default, the notebook stores the dataset and model outputs in Google Drive so you can resume later without downloading or retraining from zero.


In [ ]:
# Check PyTorch and GPU availability.
import torch
import torchvision

print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


In [ ]:
# Imports used in the notebook.
from pathlib import Path
import os
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights,
    vit_b_16,
    ViT_B_16_Weights,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)


In [ ]:
# Reproducibility: this makes repeated runs more similar.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


## 2. Download the MURA dataset

The MURA dataset is available on Kaggle as `cjinny/mura-v11`. This section downloads and unzips it.

You need a Kaggle API token:

1. Open Kaggle.
2. Go to `Account > API > Create New Token`.
3. Upload the downloaded `kaggle.json` file when this notebook asks for it.

If Kaggle gives a permission error, open the dataset page in your browser once and accept any required terms, then rerun the download cell.


In [ ]:
# Install Kaggle CLI if Colab does not already have it.
!pip -q install kaggle


In [ ]:
# Upload kaggle.json only if it is not already configured.
from google.colab import files

kaggle_json = Path('/root/.kaggle/kaggle.json')

if not kaggle_json.exists():
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('Please upload your Kaggle API file named kaggle.json')

    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    Path('kaggle.json').rename(kaggle_json)
    os.chmod(kaggle_json, 0o600)

print('Kaggle API file is ready:', kaggle_json.exists())


In [ ]:
# Download/cache MURA safely.
# We cache the zip in Google Drive, but extract to Colab local disk for faster training.
import shutil
import subprocess
import zipfile

DATASET_SLUG = 'cjinny/mura-v11'
USE_DRIVE_DATASET_CACHE = True

LOCAL_DATA_ROOT = Path('/content/data')
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

if USE_DRIVE_DATASET_CACHE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/datasets')
    DRIVE_DATA_ROOT.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_DATA_ROOT = None

local_zip = LOCAL_DATA_ROOT / 'mura-v11.zip'
drive_zip = DRIVE_DATA_ROOT / 'mura-v11.zip' if DRIVE_DATA_ROOT else None

# Step 1: get the dataset zip locally.
if not local_zip.exists():
    if drive_zip is not None and drive_zip.exists():
        print('Copying cached dataset zip from Drive to Colab local disk...')
        shutil.copy2(drive_zip, local_zip)
    else:
        print('Downloading dataset from Kaggle...')
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', str(LOCAL_DATA_ROOT)],
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(
                'Kaggle download failed. Check that kaggle.json is correct and that you accepted the dataset terms on Kaggle.'
            )

        downloaded_zips = sorted(LOCAL_DATA_ROOT.glob('*.zip'))
        if not downloaded_zips:
            raise FileNotFoundError('Kaggle finished, but no zip file was found in /content/data.')
        downloaded_zip = downloaded_zips[0]
        if downloaded_zip != local_zip:
            downloaded_zip.rename(local_zip)

        if drive_zip is not None and not drive_zip.exists():
            print('Saving dataset zip cache to Drive...')
            shutil.copy2(local_zip, drive_zip)

if not zipfile.is_zipfile(local_zip):
    if local_zip.exists():
        local_zip.unlink()
    if drive_zip is not None and drive_zip.exists():
        drive_zip.unlink()
    raise RuntimeError('The cached dataset zip was corrupted. It was deleted. Rerun this cell to download it again.')

# Step 2: extract locally if needed.
csv_files = list(LOCAL_DATA_ROOT.rglob('train_image_paths.csv'))
if not csv_files:
    print('Extracting dataset zip to Colab local disk...')
    with zipfile.ZipFile(local_zip, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_ROOT)
    csv_files = list(LOCAL_DATA_ROOT.rglob('train_image_paths.csv'))

if not csv_files:
    raise FileNotFoundError('Could not find train_image_paths.csv after extraction.')

DATA_DIR = csv_files[0].parent
print('Using Drive dataset cache:', USE_DRIVE_DATASET_CACHE)
print('Local dataset folder for training:', DATA_DIR)
print('Drive zip cache:', drive_zip)


In [ ]:
# ViT training is slow, so Drive output is recommended.
# This saves checkpoints even if the Colab session disconnects later.
USE_DRIVE_OUTPUTS = True

if USE_DRIVE_OUTPUTS:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/mura_xray_outputs')
else:
    OUTPUT_DIR = Path('/content/mura_xray_outputs')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output folder:', OUTPUT_DIR)
print('Using Google Drive outputs:', USE_DRIVE_OUTPUTS)


## 3. Configuration

These settings are intentionally small for the first run. For the final run, increase `EPOCHS` and set the sample limits to `None`.


In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 3
LEARNING_RATE = 1e-4

# ViT is much larger than MobileNet, so it usually needs a smaller fine-tuning learning rate.
MODEL_LEARNING_RATES = {
    'mobilenet_v3': 1e-4,
    'vit_b16': 1e-5,
}

# Use small limits while learning and debugging.
# Set both to None for the full experiment.
MAX_TRAIN_IMAGES = 8000
MAX_VALID_IMAGES = 2000

# Optional: train on one body part first, such as 'XR_WRIST'.
# Set to None to use all MURA body parts.
BODY_PART = None


## 4. Load MURA image paths

MURA labels are inside the folder names:

- `study..._positive` means abnormal
- `study..._negative` means normal

This notebook uses the official MURA train and valid split when the CSV files are available.


In [ ]:
def label_from_path(path):
    text = str(path).lower()
    if 'positive' in text:
        return 1
    if 'negative' in text:
        return 0
    raise ValueError(f'Cannot find label in path: {path}')


def resolve_mura_path(raw_path, data_dir):
    raw_path = Path(str(raw_path))
    if raw_path.is_absolute():
        return raw_path
    if str(raw_path).startswith('MURA-v1.1'):
        return data_dir.parent / raw_path
    return data_dir / raw_path


def load_mura_split(data_dir, split):
    csv_path = data_dir / f'{split}_image_paths.csv'

    if csv_path.exists():
        df = pd.read_csv(csv_path, header=None, names=['path'])
        df['path'] = df['path'].apply(lambda p: str(resolve_mura_path(p, data_dir)))
    else:
        image_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg']:
            image_files.extend((data_dir / split).rglob(ext))
        df = pd.DataFrame({'path': [str(p) for p in image_files]})

    df['label'] = df['path'].apply(label_from_path)

    if BODY_PART is not None:
        df = df[df['path'].str.contains(BODY_PART, case=False, regex=False)]

    return df.reset_index(drop=True)


train_df = load_mura_split(DATA_DIR, 'train')
valid_df = load_mura_split(DATA_DIR, 'valid')

def check_dataframe(df, name):
    if len(df) == 0:
        raise ValueError(f'{name} dataframe is empty. Check DATA_DIR and BODY_PART.')

    missing_paths = [p for p in df['path'] if not Path(p).exists()]
    if missing_paths:
        raise FileNotFoundError(f'{name} has missing image paths. First missing path: {missing_paths[0]}')

    labels = set(df['label'].unique().tolist())
    if labels != {0, 1}:
        raise ValueError(f'{name} should contain both labels 0 and 1, but found: {labels}')


check_dataframe(train_df, 'train')
check_dataframe(valid_df, 'valid')

print('Train images:', len(train_df))
print('Valid images:', len(valid_df))
print('\nTrain label counts:')
print(train_df['label'].value_counts().sort_index())
print('\nValid label counts:')
print(valid_df['label'].value_counts().sort_index())


In [ ]:
# Keep the first run small. This makes debugging much faster.
def limit_dataframe(df, max_rows):
    if max_rows is None or len(df) <= max_rows:
        return df
    return df.sample(n=max_rows, random_state=SEED).reset_index(drop=True)


train_df = limit_dataframe(train_df, MAX_TRAIN_IMAGES)
valid_df = limit_dataframe(valid_df, MAX_VALID_IMAGES)

print('Train images after limit:', len(train_df))
print('Valid images after limit:', len(valid_df))


## 5. Show a few examples

Always inspect the data visually before training. It helps catch path mistakes and broken images early.


In [ ]:
def show_examples(df, count=6):
    sample = df.sample(n=min(count, len(df)), random_state=SEED)
    plt.figure(figsize=(12, 6))

    for i, row in enumerate(sample.itertuples(), start=1):
        image = Image.open(row.path).convert('L')
        label = 'abnormal' if row.label == 1 else 'normal'

        plt.subplot(2, 3, i)
        plt.imshow(image, cmap='gray')
        plt.title(label)
        plt.axis('off')

    plt.tight_layout()
    plt.show()


show_examples(train_df)


## 6. Dataset and preprocessing

The images are grayscale X-rays, but ImageNet-pretrained models expect 3-channel input. We convert each image to RGB before applying transforms.


In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

valid_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class MuraDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(row['path']).convert('RGB')
        label = torch.tensor(row['label'], dtype=torch.float32)

        if self.transform is not None:
            image = self.transform(image)

        return image, label


In [ ]:
train_dataset = MuraDataset(train_df, transform=train_transforms)
valid_dataset = MuraDataset(valid_df, transform=valid_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

images, labels = next(iter(train_loader))
print('Image batch shape:', images.shape)
print('Label batch shape:', labels.shape)


## 7. Create the models

We use pretrained models from `torchvision` and only replace the final classification layer. This keeps the model code short and easy to understand.


In [ ]:
def create_model(model_name):
    if model_name == 'mobilenet_v3':
        weights = MobileNet_V3_Large_Weights.DEFAULT
        model = mobilenet_v3_large(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, 1)
        return model

    if model_name == 'vit_b16':
        weights = ViT_B_16_Weights.DEFAULT
        model = vit_b_16(weights=weights)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, 1)
        return model

    raise ValueError(f'Unknown model: {model_name}')


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_loss_fn(show_info=True):
    positive_count = int(train_df['label'].sum())
    negative_count = int(len(train_df) - positive_count)
    weight_value = negative_count / max(positive_count, 1)
    pos_weight = torch.tensor([weight_value], dtype=torch.float32, device=device)

    if show_info:
        print('Normal images:', negative_count)
        print('Abnormal images:', positive_count)
        print('Positive class weight:', round(weight_value, 4))

    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)


## 8. Training and evaluation functions

The model outputs one number called a logit. After `sigmoid`, it becomes the probability of the abnormal class.


In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    all_probs = []
    all_labels = []

    for images, labels in tqdm(loader, desc='Training'):
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy().ravel())

    avg_loss = total_loss / len(loader.dataset)
    preds = (np.array(all_probs) >= 0.5).astype(int)
    labels = np.array(all_labels).astype(int)
    accuracy = accuracy_score(labels, preds)

    return avg_loss, accuracy


def find_best_threshold(labels, probs):
    thresholds = np.arange(0.05, 0.96, 0.01)
    f1_scores = []

    for threshold in thresholds:
        preds = (probs >= threshold).astype(int)
        f1_scores.append(f1_score(labels, preds, zero_division=0))

    best_index = int(np.argmax(f1_scores))
    return float(thresholds[best_index])


def calculate_metrics(labels, probs, threshold):
    preds = (probs >= threshold).astype(int)
    metrics = {
        'threshold': threshold,
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
    }

    try:
        metrics['roc_auc'] = roc_auc_score(labels, probs)
    except ValueError:
        metrics['roc_auc'] = np.nan

    return metrics, preds


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    all_probs = []
    all_labels = []

    for images, labels in tqdm(loader, desc='Evaluating'):
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(images)
        loss = loss_fn(logits, labels)

        total_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy().ravel())

    labels = np.array(all_labels).astype(int)
    probs = np.array(all_probs)
    threshold = find_best_threshold(labels, probs)
    metrics, preds = calculate_metrics(labels, probs, threshold)
    metrics['loss'] = total_loss / len(loader.dataset)

    return metrics, labels, preds, probs


In [ ]:
def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def train_model(model_name, resume=True):
    print(f'\nStarting model: {model_name}')
    model = create_model(model_name).to(device)
    print('Trainable parameters:', count_parameters(model))

    loss_fn = make_loss_fn(show_info=True)
    learning_rate = MODEL_LEARNING_RATES.get(model_name, LEARNING_RATE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    print('Learning rate:', learning_rate)

    best_f1 = -1
    start_epoch = 1
    history = []

    best_path = OUTPUT_DIR / f'{model_name}_best.pt'
    last_path = OUTPUT_DIR / f'{model_name}_last_checkpoint.pt'
    history_path = OUTPUT_DIR / f'{model_name}_history.csv'

    if resume and last_path.exists():
        checkpoint = load_checkpoint(last_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        for group in optimizer.param_groups:
            group['lr'] = learning_rate
        start_epoch = checkpoint['epoch'] + 1
        best_f1 = checkpoint.get('best_f1', -1)
        history = checkpoint.get('history', [])
        print('Resumed from:', last_path)
        print('Next epoch:', start_epoch)

    if start_epoch > EPOCHS:
        print('Training already reached EPOCHS. Increase EPOCHS if you want to continue.')
    else:
        for epoch in range(start_epoch, EPOCHS + 1):
            print(f'\nEpoch {epoch}/{EPOCHS}')
            train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, loss_fn)
            val_metrics, _, _, _ = evaluate(model, valid_loader, loss_fn)

            row = {
                'epoch': epoch,
                'train_loss': train_loss,
                'train_accuracy': train_accuracy,
                **{f'val_{k}': v for k, v in val_metrics.items()},
            }
            history.append(row)
            print(row)

            if val_metrics['f1'] > best_f1:
                best_f1 = val_metrics['f1']
                torch.save(model.state_dict(), best_path)
                print('Saved best model:', best_path)

            checkpoint = {
                'epoch': epoch,
                'model_name': model_name,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
                'history': history,
            }
            torch.save(checkpoint, last_path)
            pd.DataFrame(history).to_csv(history_path, index=False)
            print('Saved latest checkpoint:', last_path)
            print('Saved history:', history_path)

    if best_path.exists():
        model.load_state_dict(load_checkpoint(best_path))
    return model, pd.DataFrame(history), best_path


## 9. Train MobileNetV3

MobileNetV3 should train faster. Run this model first to confirm the pipeline works.


In [ ]:
mobilenet_model, mobilenet_history, mobilenet_path = train_model('mobilenet_v3')
mobilenet_history


## 10. Train Vision Transformer

ViT is heavier than MobileNetV3. Keep the documented `BATCH_SIZE = 32` unless Colab gives an out-of-memory error. If that happens, lower the batch size and mention the change in the report.


In [ ]:
vit_model, vit_history, vit_path = train_model('vit_b16')
vit_history


## 11. Final evaluation helpers

Here we measure metrics, inference time, and model size for the final comparison table.


In [ ]:
@torch.no_grad()
def measure_inference_time(model, loader, batches=20):
    model.eval()
    times = []

    for batch_index, (images, _) in enumerate(loader):
        if batch_index >= batches:
            break

        images = images.to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()

        start = time.perf_counter()
        _ = model(images)

        if device.type == 'cuda':
            torch.cuda.synchronize()

        seconds = time.perf_counter() - start
        times.append(seconds / images.size(0))

    return float(np.mean(times))


def model_file_size_mb(path):
    return Path(path).stat().st_size / (1024 * 1024)


def summarize_model(model_name, model, checkpoint_path):
    loss_fn = make_loss_fn(show_info=False)
    metrics, labels, preds, probs = evaluate(model, valid_loader, loss_fn)
    metrics['model'] = model_name
    metrics['parameters'] = count_parameters(model)
    metrics['model_size_mb'] = model_file_size_mb(checkpoint_path)
    metrics['inference_ms_per_image'] = measure_inference_time(model, valid_loader) * 1000
    return metrics, labels, preds, probs


In [ ]:
mobilenet_summary, mobilenet_labels, mobilenet_preds, mobilenet_probs = summarize_model(
    'MobileNetV3', mobilenet_model, mobilenet_path
)

vit_summary, vit_labels, vit_preds, vit_probs = summarize_model(
    'ViT-B/16', vit_model, vit_path
)

comparison_df = pd.DataFrame([mobilenet_summary, vit_summary])
comparison_df = comparison_df[[
    'model',
    'accuracy',
    'precision',
    'recall',
    'f1',
    'threshold',
    'roc_auc',
    'inference_ms_per_image',
    'model_size_mb',
    'parameters',
]]

comparison_df


In [ ]:
# Save the comparison table for your report.
comparison_path = OUTPUT_DIR / 'mobilenet_vs_vit_metrics.csv'
comparison_df.to_csv(comparison_path, index=False)
print('Saved:', comparison_path)


## 12. Confusion matrices

A confusion matrix shows which cases were correctly or incorrectly classified.


In [ ]:
def plot_confusion_matrix(labels, preds, title):
    matrix = confusion_matrix(labels, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['normal', 'abnormal'],
        yticklabels=['normal', 'abnormal'],
    )
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.show()


plot_confusion_matrix(mobilenet_labels, mobilenet_preds, 'MobileNetV3 Confusion Matrix')
plot_confusion_matrix(vit_labels, vit_preds, 'ViT-B/16 Confusion Matrix')


## 13. Predict one image

This is the simple prediction function you can later reuse in an API or agent project.


In [ ]:
@torch.no_grad()
def predict_image(model, image_path, threshold=0.5):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    tensor = valid_transforms(image).unsqueeze(0).to(device)

    probability_abnormal = torch.sigmoid(model(tensor)).item()
    predicted_label = 'abnormal' if probability_abnormal >= threshold else 'normal'
    confidence = probability_abnormal if predicted_label == 'abnormal' else 1 - probability_abnormal

    return {
        'prediction': predicted_label,
        'confidence': confidence,
        'threshold': threshold,
        'probability_normal': 1 - probability_abnormal,
        'probability_abnormal': probability_abnormal,
    }


example_path = valid_df.iloc[0]['path']
vit_threshold = float(comparison_df.loc[comparison_df['model'] == 'ViT-B/16', 'threshold'].iloc[0])
print('Example image:', example_path)
predict_image(vit_model, example_path, threshold=vit_threshold)


## 14. Download or reuse the trained models

The best checkpoints are saved in `OUTPUT_DIR`:

- `mobilenet_v3_best.pt`
- `vit_b16_best.pt`
- `mobilenet_v3_last_checkpoint.pt`
- `vit_b16_last_checkpoint.pt`

The `best.pt` files are for inference. The `last_checkpoint.pt` files are for resuming training because they include the optimizer state, epoch number, and history.

This section creates one zip file so you can download the trained models and metrics from Colab.


In [ ]:
print('MobileNet checkpoint:', mobilenet_path)
print('ViT checkpoint:', vit_path)
print('Metrics CSV:', comparison_path)

import shutil

zip_path = Path('/content/mura_xray_outputs.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive('/content/mura_xray_outputs', 'zip', OUTPUT_DIR)

print('Created zip file:', zip_path)

# Uncomment this line if you want Colab to start downloading the zip automatically.
# files.download(str(zip_path))


## 15. Notes for the later medical AI agent

This notebook produces simple discriminative models. Later, the portfolio agent can use these saved models as tools:

```text
Input X-ray image
-> MobileNetV3 prediction
-> ViT prediction
-> optional medical foundation model check
-> MedGemma report generator
-> final structured report
```

For now, keep the internship deliverable focused on the required MobileNetV3 vs ViT comparison.
